In [53]:
import pandas as pd
from pathlib import Path

Bronze_file = Path("bronze_layer")/"araz_bronze_2026-02-13_2026-02-13T20-02-04Z.csv"

df=pd.read_csv(Bronze_file,encoding="utf-8-sig")

print(len(df))

df.head(1)

348


,scrape_run_id,scrape_ts_utc,snapshot_date,market,category_raw,product_name_raw,barcode,price_current_azn,price_old_azn,price_current_raw,price_old_raw,promo_flag,purchasable_balance,availability_status,unit_info_raw,product_hash,dq_missing_barcode,dq_invalid_price,dq_missing_purchasable_balance
0,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13T20:02:04.946584,2026-02-13,Araz,edviyat,Mastero Xörək Duzu Yodlaşdırılmış 1000 qr,4.760106e+12,0.7,0.7,70,70,0,24,available,1 əd.,a99d270173d33cbde3944382639adf604c028811d5bbd6...,0,0,0


In [54]:
df['scrape_run_id'].isnull().sum()
df['market'].isnull().sum()
df['category_raw'].isnull().sum()
df['product_name_raw'].isnull().sum()

np.int64(0)

In [55]:
df["scrape_ts_utc"].dtype
df["snapshot_date"].dtype
df["scrape_ts_utc"]=pd.to_datetime(df["scrape_ts_utc"],errors="coerce")
df["snapshot_date"]=pd.to_datetime(df["snapshot_date"],errors="coerce").dt.date
df["scrape_ts"]=pd.to_datetime(df["scrape_ts_utc"],errors="coerce",utc=True)
df["scrape_ts_baku"]=df["scrape_ts"].dt.tz_convert("Asia/Baku")
df["baku_hour"]=df["scrape_ts_baku"].dt.hour

df["session"]=None

df.loc[df["baku_hour"].between(6,11),"session"]="morning"
df.loc[df["baku_hour"].between(18,23),"session"]="evening"

df["market_day"]=df["market"].astype(str)+"|"+df["snapshot_date"].astype(str)
df["market_day_session"] = df["market_day"].astype(str) + "|" + df["session"].astype(str)

In [ ]:
import re
import numpy as np
import pandas as pd


UNIT_RE = re.compile(
    r"(\d+(?:[.,]\d+)?)\s*(kq|kg|qram|qr|gr|g|ml|l|lt|litr)\b",
    re.I
)

MULTI_RE = re.compile(
    r"(\d+)\s*[xX*]\s*(\d+(?:[.,]\d+)?)\s*(kq|kg|qram|qr|gr|g|ml|l|lt|litr)\b",
    re.I
)

MULTI_REV_RE = re.compile(
    r"(\d+(?:[.,]\d+)?)\s*(kq|kg|qram|qr|gr|g|ml|l|lt|litr)\s*[xX*]\s*(\d+)\b",
    re.I
)



def parse_num(s: str) -> float:
    s = s.strip()

    
    if re.match(r"^\d+[.,]\d{3}$", s):
        s = s.replace(".", "").replace(",", "")
        return float(s)

    
    return float(s.replace(",", "."))



def to_std(qty, unit):
    unit = unit.lower()

    if unit in ["kq", "kg"]:
        return qty * 1000, "g"

    if unit in ["qram", "qr", "gr", "g"]:
        return qty, "g"

    if unit in ["l", "lt", "litr"]:
        return qty * 1000, "ml"

    if unit == "ml":
        return qty, "ml"

    return np.nan, None



def extract_std_qty(text):

    if not isinstance(text, str):
        return np.nan, None, 1

    t = text.lower()

    
    m = MULTI_RE.search(t)
    if m:
        count = int(m.group(1))
        qty = parse_num(m.group(2))
        unit = m.group(3)

        std_qty, std_unit = to_std(qty * count, unit)
        return std_qty, std_unit, count

    
    m = MULTI_REV_RE.search(t)
    if m:
        qty = parse_num(m.group(1))
        unit = m.group(2)
        count = int(m.group(3))

        std_qty, std_unit = to_std(qty * count, unit)
        return std_qty, std_unit, count

    all_matches = UNIT_RE.findall(t)

    if not all_matches:
        return np.nan, None, 1

    qty_str, unit = all_matches[-1]
    qty = parse_num(qty_str)

    std_qty, std_unit = to_std(qty, unit)

    return std_qty, std_unit, 1



df["product_name_raw"] = df["product_name_raw"].astype("string")

df[["std_qty_total", "std_unit", "pack_multiplier"]] = df["product_name_raw"].apply(
    lambda x: pd.Series(extract_std_qty(str(x) if pd.notna(x) else None))
)



df["dq_missing_unit"] = (
    df["std_qty_total"].isna() | (df["std_qty_total"] <= 0)
).astype(int)


df["unit_price_1kg"] = np.where(
    (df["std_unit"] == "g") & (df["dq_missing_unit"] == 0),
    df["price_current_azn"] / (df["std_qty_total"] / 1000),
    np.nan
)

df["unit_price_1l"] = np.where(
    (df["std_unit"] == "ml") & (df["dq_missing_unit"] == 0),
    df["price_current_azn"] / (df["std_qty_total"] / 1000),
    np.nan
)


df["unit_price_raw"] = np.where(
    df["dq_missing_unit"] == 1,
    np.nan,
    df["price_current_azn"] / df["std_qty_total"]
)

In [ ]:
# Default 1 kg
mask = (
    df["std_qty_total"].isna() &
    df["product_name_raw"].str.contains(r"\b(kq|kg)\b", case=False, na=False)
)

df.loc[mask, "std_qty_total"] = 1000
df.loc[mask, "std_unit"] = "g"
df.loc[mask, "pack_multiplier"] = 1

C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\1761006126.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["product_name_raw"].str.contains(r"\b(kq|kg)\b", case=False, na=False)


In [60]:
df = df[~df["product_name_raw"].str.contains("yumurta", case=False, na=False)]

In [61]:
df["unit_price_base"] = np.where(
    df["std_unit"]=="g", df["unit_price_1kg"],
    np.where(df["std_unit"]=="ml", df["unit_price_1l"], np.nan)
)


In [62]:
df["unit_price_base_label"] = np.where(
    df["std_unit"]=="g", "AZN / kg",
    np.where(df["std_unit"]=="ml", "AZN / L", pd.NA)
)

In [ ]:
import re
import numpy as np
import pandas as pd

df["name_lc"] = df["product_name_raw"].astype("string").str.lower()

SUB_RULES = [

    
    ("Yuyucu toz",  r"\b(yuyucu\s*toz|toz\s*deterjan|deterjan)\b"),
    ("Yuyucu maye",   r"\b(yuyucu|perwoll|vanish)\b"),

    ("Şəkər", r"\b(şəkər|qənd|kəllə\s*qənd|nabat|noğul)\b"),
    ("Duz",   r"\b(duz|xörək\s*duzu|limon\s*duzu|dəniz\s*duzu|turşu\s*duzu)\b"),

    ("Un",     r"\b(un|buğda\s*unu|qarabaşaq\s*unu|misir\s*unu|nişasta|nishasta)\b"),
    ("Düyü",   r"\b(düyü|duyu|basmati|sella|qarabaşaq)\b"),
    ("Makaron",r"\b(makaron|spagetti|əriştə|vermişel|vermeşel|penne|fusilli|tel\s*şehriye|burgu|qələm|boru)\b"),

    ("Süd",    r"\b(süd|sud|milk)\b"),
    ("Qatıq",  r"\b(qatıq|qatiq)\b"),
    ("Ayran",  r"\bayran\b"),

    ("Kərə yağı",     r"\b(kərə\s*yağı|kərəyağı|eridilmiş\s*yağ)\b"),
    ("Zeytun yağı",   r"\b(zeytun\s*yağı)\b"),
    ("Günəbaxan yağı",r"\b(günəbaxan\s*yağı)\b"),
    ("Qarğıdalı yağı",r"\b(qarğıdalı\s*yağı|qargidali|misir\s*yağı)\b"),
]

df["category_sub"] = "Others"

for sub, pattern in SUB_RULES:
    mask = df["name_lc"].str.contains(pattern, regex=True, na=False)
    df.loc[mask & (df["category_sub"] == "Others"), "category_sub"] = sub


GROUP_MAP = {
    
    "Un": "Core staples",
    "Düyü": "Core staples",
    "Makaron": "Core staples",

    "Süd": "Dairy",
    "Qatıq": "Dairy",
    "Ayran": "Dairy",

    "Kərə yağı": "Cooking fats",
    "Zeytun yağı": "Cooking fats",
    "Günəbaxan yağı": "Cooking fats",
    "Qarğıdalı yağı": "Cooking fats",

    "Şəkər": "Everyday essentials",
    "Duz": "Everyday essentials",

    "Yuyucu toz": "Non food",
    "Yuyucu maye": "Non food",
}

df["category_group"] = df["category_sub"].map(GROUP_MAP).fillna("Others")

print("Others count:", (df["category_sub"] == "Others").sum())

Others count: 205


C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\2154251171.py:38: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df["name_lc"].str.contains(pattern, regex=True, na=False)
C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\2154251171.py:38: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df["name_lc"].str.contains(pattern, regex=True, na=False)
C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\2154251171.py:38: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df["name_lc"].str.contains(pattern, regex=True, na=False)
C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\2154251171.py:38: UserWarning: This pattern is interpreted as a regular expression, and has match grou

In [ ]:
import numpy as np
import pandas as pd


FAT_FIX_RULES = [
    ("Günəbaxan yağı", r"g[üu]n[əe]baxan.*ya[gğ][ıi]"),
    ("Zeytun yağı",    r"zeytun.*ya[gğ][ıi]"),
    ("Qarğıdalı yağı", r"(qar[ğı]dal[ıi].*ya[gğ][ıi]|misir.*ya[gğ][ıi])"),
]


if "name_lc" not in df.columns:
    df["name_lc"] = df["product_name_raw"].astype("string").str.lower()

for sub, pattern in FAT_FIX_RULES:
    mask = (df["category_sub"] == "Others") & df["name_lc"].str.contains(pattern, regex=True, na=False)
    df.loc[mask, "category_sub"] = sub


GROUP_MAP_FIX = {
    "Günəbaxan yağı": "Cooking fats",
    "Zeytun yağı": "Cooking fats",
    "Qarğıdalı yağı": "Cooking fats",
}
df.loc[df["category_sub"].isin(GROUP_MAP_FIX.keys()), "category_group"] = df["category_sub"].map(GROUP_MAP_FIX)

print("Others count after fats fix:", (df["category_sub"] == "Others").sum())
df.loc[df["category_sub"] == "Others", "product_name_raw"].head(30)

Others count after fats fix: 176


C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\2451582572.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = (df["category_sub"] == "Others") & df["name_lc"].str.contains(pattern, regex=True, na=False)


4              Galina Blanca Göbələk 10qr
5     Ilgim Sarıkök Salafan Paket 15*40qr
6                 Limon Tozu 15qr Soyigit
7        Nid Uyudulmuş Sarımsaq Tozu 15qr
8              Nid Şokolad Damlaları 20qr
9                  Nid Xmeli Suneli 25 Gr
10             Acı Pul Bibər 15qr Soyigit
11     Acı Toz Qırmızı Bibər 15qr Soyigit
12          Nid Şəkərli Muncuq Paket 10qr
14                   Ilğım Pul Bibər 50qr
15      Misir Nishastasi 12x200qr Soyigit
16                 Nid Dəfnə Yarpağı 10qr
17              Nid Kartof Nisastasi 70qr
18                     Nid Quru Nane 50qr
20                      Ilgim Zirinc 50qr
21                     Nid Kəklikotu 50qr
22                     Kent Brinqer Vanil
23                Kent Kabartma Tozu 10qr
24                Ilgim Şəkərli Vanil 5qr
25                    Azerduz Paket 750gr
26               Ilgim Qabartma Tozu 10qr
27                    Kent Ət Bulyon 10qr
28                        Şəkərli Vanilin
29          Hamur Qabartma Tozu 5-

In [ ]:
name_lc = df["product_name_raw"].astype("string").str.lower()


mask_kere = (df["category_sub"]=="Others") & (
    name_lc.str.contains("eridil", na=False) &
    name_lc.str.contains("ya", na=False)
)


mask_qargidali = (df["category_sub"]=="Others") & (
    (name_lc.str.contains("qar", na=False) | name_lc.str.contains("qargid", na=False)) &
    name_lc.str.contains("ya", na=False)
)

mask_gunebaxan = (df["category_sub"]=="Others") & (
    (name_lc.str.contains("gunebaxan", na=False) | name_lc.str.contains("günəbaxan", na=False)) &
    name_lc.str.contains("ya", na=False)
)


df.loc[mask_kere, "category_sub"] = "Kərə yağı"
df.loc[mask_qargidali, "category_sub"] = "Qarğıdalı yağı"
df.loc[mask_gunebaxan, "category_sub"] = "Günəbaxan yağı"

df.loc[df["category_sub"].isin(["Kərə yağı","Qarğıdalı yağı","Günəbaxan yağı"]), "category_group"] = "Cooking fats"

print("Others count:", (df["category_sub"]=="Others").sum())

Others count: 156


In [ ]:
name_lc = df["product_name_raw"].astype("string").str.lower()

mask_duz_fix = (df["category_sub"]=="Others") & (
    name_lc.str.contains(r"\baz[əe]rduz\b|\bazerduz\b|\bduz\b|x[oö]rək\s*duzu|süfrə\s*duzu|dəniz\s*duzu|turşu\s*duzu|limon\s*duzu",
                         regex=True, na=False)
)
df.loc[mask_duz_fix, "category_sub"] = "Duz"
df.loc[df["category_sub"]=="Duz", "category_group"] = "Everyday essentials"

mask_un_fix = (df["category_sub"]=="Others") & (
    name_lc.str.contains(r"\b(un|unu|buğda\s*unu|misir\s*unu|qarabaşaq\s*unu|nişasta|nishasta)\b",
                         regex=True, na=False)
)
df.loc[mask_un_fix, "category_sub"] = "Un"
df.loc[df["category_sub"]=="Un", "category_group"] = "Core staples"

mask_duyu_fix = (df["category_sub"]=="Others") & (
    name_lc.str.contains(r"\b(düyü|duyu|sella|basmat[ıi]|basmati)\b", regex=True, na=False)
)
df.loc[mask_duyu_fix, "category_sub"] = "Düyü"
df.loc[df["category_sub"]=="Düyü", "category_group"] = "Core staples"

print("Others count:", (df["category_sub"]=="Others").sum())

Others count: 147


C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\1888730186.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  name_lc.str.contains(r"\b(un|unu|buğda\s*unu|misir\s*unu|qarabaşaq\s*unu|nişasta|nishasta)\b",
C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\1888730186.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  name_lc.str.contains(r"\b(düyü|duyu|sella|basmat[ıi]|basmati)\b", regex=True, na=False)


In [ ]:
keys = ["makaronduyu", "yaglar"]

hit_cols = []
for c in df.columns:
    s = df[c].astype(str).str.lower()
    if any(s.str.contains(k, na=False).any() for k in keys):
        hit_cols.append(c)

hit_cols

['category_raw']

In [ ]:
cat_col = hit_cols[0]   
s = df['category_raw'].astype(str).str.lower()

mask_other = df["category_sub"].eq("Others")

mask_duyu   = mask_other & s.str.contains("duyu", na=False)
mask_makaron= mask_other & s.str.contains("makaron", na=False)

df.loc[mask_duyu, "category_sub"]   = "Düyü"
df.loc[mask_duyu, "category_group"] = "Core staples"

df.loc[mask_makaron, "category_sub"]   = "Makaron"
df.loc[mask_makaron, "category_group"] = "Core staples"

print("Others count:", (df["category_sub"]=="Others").sum())

Others count: 126


In [ ]:

df["dq_missing_barcode"] = df["barcode"].isna().astype(int)

df["barcode_str"] = df["barcode"].apply(
    lambda x: str(int(x)) if pd.notna(x) else None
)


In [70]:
others_df = df[df["category_sub"] == "Others"]
display(others_df)

,scrape_run_id,scrape_ts_utc,snapshot_date,market,category_raw,product_name_raw,barcode,price_current_azn,price_old_azn,price_current_raw,...,dq_missing_unit,unit_price_1kg,unit_price_1l,unit_price_raw,unit_price_base,unit_price_base_label,name_lc,category_sub,category_group,barcode_str
4,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:04.946584,2026-02-13,Araz,edviyat,Galina Blanca Göbələk 10qr,8.410309e+07,0.24,0.24,24,...,0,24.000000,NaN,0.024000,24.000000,AZN / kg,galina blanca göbələk 10qr,Others,Others,84103093
5,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:04.946584,2026-02-13,Araz,edviyat,Ilgim Sarıkök Salafan Paket 15*40qr,4.760095e+12,2.24,2.24,224,...,0,3.733333,NaN,0.003733,3.733333,AZN / kg,ilgim sarıkök salafan paket 15*40qr,Others,Others,4760095013175
6,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:04.946584,2026-02-13,Araz,edviyat,Limon Tozu 15qr Soyigit,8.694587e+12,0.59,0.90,59,...,0,39.333333,NaN,0.039333,39.333333,AZN / kg,limon tozu 15qr soyigit,Others,Others,8694587101957
7,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:04.946584,2026-02-13,Araz,edviyat,Nid Uyudulmuş Sarımsaq Tozu 15qr,4.760081e+12,1.15,1.15,115,...,0,76.666667,NaN,0.076667,76.666667,AZN / kg,nid uyudulmuş sarımsaq tozu 15qr,Others,Others,4760080701391
8,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:04.946584,2026-02-13,Araz,edviyat,Nid Şokolad Damlaları 20qr,4.760081e+12,1.00,1.00,100,...,0,50.000000,NaN,0.050000,50.000000,AZN / kg,nid şokolad damlaları 20qr,Others,Others,4760080701834
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.127228,2026-02-13,Araz,temizlik,Age Qab Yuyan Sf 12* 1 Lt,4.760091e+12,1.35,1.35,135,...,0,NaN,0.112500,0.000113,0.112500,AZN / L,age qab yuyan sf 12* 1 lt,Others,Others,4760090601179
244,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.127228,2026-02-13,Araz,temizlik,Crystalex Qabyuyan Maye Üzüm Gücü 450ml,4.760093e+12,2.49,3.35,249,...,0,NaN,5.533333,0.005533,5.533333,AZN / L,crystalex qabyuyan maye üzüm gücü 450ml,Others,Others,4760092804004
245,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.127228,2026-02-13,Araz,temizlik,Bloom Qab Yuyan Maye 4L,4.760091e+12,3.39,4.15,339,...,0,NaN,0.847500,0.000848,0.847500,AZN / L,bloom qab yuyan maye 4l,Others,Others,4760090601742
264,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.155162,2026-02-13,Araz,un,Pudov Solod 300qr,4.607012e+12,2.70,2.70,270,...,0,9.000000,NaN,0.009000,9.000000,AZN / kg,pudov solod 300qr,Others,Others,4607012293169


In [ ]:
indices_to_drop = [61,62]
df = df.drop(indices_to_drop)

In [ ]:
df["name_lc"] = df["product_name_raw"].astype("string").str.lower()

print("şəkər var?:", df["name_lc"].str.contains("şəkər", na=False).sum())
print("seker var?:", df["name_lc"].str.contains("seker", na=False).sum())

display(df.loc[df["category_sub"].eq("Others"), ["product_name_raw","name_lc","category_raw"]].head(20))

şəkər var?: 27
seker var?: 0


,product_name_raw,name_lc,category_raw
4,Galina Blanca Göbələk 10qr,galina blanca göbələk 10qr,edviyat
5,Ilgim Sarıkök Salafan Paket 15*40qr,ilgim sarıkök salafan paket 15*40qr,edviyat
6,Limon Tozu 15qr Soyigit,limon tozu 15qr soyigit,edviyat
7,Nid Uyudulmuş Sarımsaq Tozu 15qr,nid uyudulmuş sarımsaq tozu 15qr,edviyat
8,Nid Şokolad Damlaları 20qr,nid şokolad damlaları 20qr,edviyat
9,Nid Xmeli Suneli 25 Gr,nid xmeli suneli 25 gr,edviyat
10,Acı Pul Bibər 15qr Soyigit,acı pul bibər 15qr soyigit,edviyat
11,Acı Toz Qırmızı Bibər 15qr Soyigit,acı toz qırmızı bibər 15qr soyigit,edviyat
12,Nid Şəkərli Muncuq Paket 10qr,nid şəkərli muncuq paket 10qr,edviyat
14,Ilğım Pul Bibər 50qr,ilğım pul bibər 50qr,edviyat


In [ ]:
import numpy as np
import pandas as pd

df["name_lc"] = df["product_name_raw"].astype("string").str.lower()
df["cat_lc"]  = df["category_raw"].astype("string").str.lower().str.strip()

df["category_sub"] = "Others"

SUB_RULES = [
    
    ("Yuyucu toz",  r"\b(yuyucu\s*toz|toz\s*deterjan|deterjan)\b"),
    ("Yuyucu maye", r"\b(yuyucu\s*maye|qabyuyan\s*maye|qabyuyan|perwoll|vanish)\b"),

    
    ("Şəkər", r"\b(şəkər|qənd|kəllə\s*qənd|nabat|noğul)\b"),
    ("Duz",   r"\b(duz|xörək\s*duzu|limon\s*duzu|dəniz\s*duzu|turşu\s*duzu)\b"),

    
    ("Un",     r"\b(un|buğda\s*unu|qarabaşaq\s*unu|misir\s*unu|nişasta|nishasta)\b"),
    ("Düyü",   r"\b(düyü|duyu|basmati|sella|qarabaşaq)\b"),
    ("Makaron",r"\b(makaron|spagetti|əriştə|vermişel|vermeşel|penne|fusilli|tel\s*şehriye|burgu|qələm|boru)\b"),

    ("Süd",    r"\b(süd|sud|milk)\b"),
    ("Qatıq",  r"\b(qatıq|qatiq|yoqurt|yogurt)\b"),
    ("Ayran",  r"\b(ayran)\b"),

    ("Kərə yağı",      r"\b(kərə\s*yağı|kərəyağı|eridilmiş\s*yağ)\b"),
    ("Zeytun yağı",    r"\b(zeytun\s*yağı|olive)\b"),
    ("Günəbaxan yağı", r"\b(günəbaxan\s*yağı|gunebaxan)\b"),
    ("Qarğıdalı yağı", r"\b(qarğıdalı\s*yağı|qargidali|misir\s*yağı)\b"),
]

for sub, pattern in SUB_RULES:
    m = df["name_lc"].str.contains(pattern, regex=True, na=False)
    df.loc[m & df["category_sub"].eq("Others"), "category_sub"] = sub


df.loc[df["category_sub"].eq("Others") & df["cat_lc"].eq("seker"), "category_sub"] = "Şəkər"


GROUP_MAP = {
    "Un": "Core staples",
    "Düyü": "Core staples",
    "Makaron": "Core staples",

    "Süd": "Dairy",
    "Qatıq": "Dairy",
    "Ayran": "Dairy",

    "Kərə yağı": "Cooking fats",
    "Zeytun yağı": "Cooking fats",
    "Günəbaxan yağı": "Cooking fats",
    "Qarğıdalı yağı": "Cooking fats",

    "Şəkər": "Everyday essentials",
    "Duz": "Everyday essentials",

    "Yuyucu toz": "Non food",
    "Yuyucu maye": "Non food",
}

df["category_group"] = df["category_sub"].map(GROUP_MAP).fillna("Others")

print("Others count:", df["category_sub"].eq("Others").sum())
print(df["category_sub"].value_counts())

Others count: 172
category_sub
Others            172
Şəkər              46
Makaron            33
Un                 22
Yuyucu maye        13
Süd                11
Duz                10
Qatıq              10
Yuyucu toz         10
Düyü                8
Qarğıdalı yağı      1
Name: count, dtype: int64


C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\1025955326.py:41: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m = df["name_lc"].str.contains(pattern, regex=True, na=False)
C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\1025955326.py:41: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m = df["name_lc"].str.contains(pattern, regex=True, na=False)
C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\1025955326.py:41: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m = df["name_lc"].str.contains(pattern, regex=True, na=False)
C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\1025955326.py:41: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To ac

In [73]:
others_df = df[df["category_sub"] == "Others"]
display(others_df)

,scrape_run_id,scrape_ts_utc,snapshot_date,market,category_raw,product_name_raw,barcode,price_current_azn,price_old_azn,price_current_raw,...,unit_price_1kg,unit_price_1l,unit_price_raw,unit_price_base,unit_price_base_label,name_lc,category_sub,category_group,barcode_str,cat_lc
4,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:04.946584,2026-02-13,Araz,edviyat,Galina Blanca Göbələk 10qr,8.410309e+07,0.24,0.24,24,...,24.000000,NaN,0.024000,24.000000,AZN / kg,galina blanca göbələk 10qr,Others,Others,84103093,edviyat
5,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:04.946584,2026-02-13,Araz,edviyat,Ilgim Sarıkök Salafan Paket 15*40qr,4.760095e+12,2.24,2.24,224,...,3.733333,NaN,0.003733,3.733333,AZN / kg,ilgim sarıkök salafan paket 15*40qr,Others,Others,4760095013175,edviyat
6,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:04.946584,2026-02-13,Araz,edviyat,Limon Tozu 15qr Soyigit,8.694587e+12,0.59,0.90,59,...,39.333333,NaN,0.039333,39.333333,AZN / kg,limon tozu 15qr soyigit,Others,Others,8694587101957,edviyat
7,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:04.946584,2026-02-13,Araz,edviyat,Nid Uyudulmuş Sarımsaq Tozu 15qr,4.760081e+12,1.15,1.15,115,...,76.666667,NaN,0.076667,76.666667,AZN / kg,nid uyudulmuş sarımsaq tozu 15qr,Others,Others,4760080701391,edviyat
8,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:04.946584,2026-02-13,Araz,edviyat,Nid Şokolad Damlaları 20qr,4.760081e+12,1.00,1.00,100,...,50.000000,NaN,0.050000,50.000000,AZN / kg,nid şokolad damlaları 20qr,Others,Others,4760080701834,edviyat
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.193487,2026-02-13,Araz,yaglar,Saratovskoye Günəbaxan Yağı 0.9 L,4.603601e+12,4.20,4.20,420,...,NaN,4.666667,0.004667,4.666667,AZN / L,saratovskoye günəbaxan yağı 0.9 l,Others,Others,4603601000187,yaglar
332,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.193487,2026-02-13,Araz,yaglar,Yurdum Günəbaxan Yağı 1lt,4.760061e+12,4.35,4.35,435,...,NaN,4.350000,0.004350,4.350000,AZN / L,yurdum günəbaxan yağı 1lt,Others,Others,4760060607989,yaglar
333,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.193487,2026-02-13,Araz,yaglar,Final Günəbaxan Yağı 0.5lt,4.760061e+12,2.90,2.90,290,...,NaN,5.800000,0.005800,5.800000,AZN / L,final günəbaxan yağı 0.5lt,Others,Others,4760060600416,yaglar
334,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.193487,2026-02-13,Araz,yaglar,Saratovskoye Günəbaxan Yağı 4.5 L,4.603601e+12,20.30,20.30,2030,...,NaN,4.511111,0.004511,4.511111,AZN / L,saratovskoye günəbaxan yağı 4.5 l,Others,Others,4603601111722,yaglar


In [ ]:
name_lc = df["product_name_raw"].astype("string").str.lower()


mask_duz_fix = (df["category_sub"]=="Others") & (
    name_lc.str.contains(r"\baz[əe]rduz\b|\bazerduz\b|\bduz\b|x[oö]rək\s*duzu|süfrə\s*duzu|dəniz\s*duzu|turşu\s*duzu|limon\s*duzu",
                         regex=True, na=False)
)
df.loc[mask_duz_fix, "category_sub"] = "Duz"
df.loc[df["category_sub"]=="Duz", "category_group"] = "Everyday essentials"


mask_un_fix = (df["category_sub"]=="Others") & (
    name_lc.str.contains(r"\b(un|unu|buğda\s*unu|misir\s*unu|qarabaşaq\s*unu|nişasta|nishasta)\b",
                         regex=True, na=False)
)
df.loc[mask_un_fix, "category_sub"] = "Un"
df.loc[df["category_sub"]=="Un", "category_group"] = "Core staples"


mask_duyu_fix = (df["category_sub"]=="Others") & (
    name_lc.str.contains(r"\b(düyü|duyu|sella|basmat[ıi]|basmati)\b", regex=True, na=False)
)
df.loc[mask_duyu_fix, "category_sub"] = "Düyü"
df.loc[df["category_sub"]=="Düyü", "category_group"] = "Core staples"

print("Others count:", (df["category_sub"]=="Others").sum())

Others count: 163


C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\1888730186.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  name_lc.str.contains(r"\b(un|unu|buğda\s*unu|misir\s*unu|qarabaşaq\s*unu|nişasta|nishasta)\b",
C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\1888730186.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  name_lc.str.contains(r"\b(düyü|duyu|sella|basmat[ıi]|basmati)\b", regex=True, na=False)


In [ ]:
name_lc = df["product_name_raw"].astype("string").str.lower()


mask_kere = (df["category_sub"]=="Others") & (
    name_lc.str.contains("eridil", na=False) &
    name_lc.str.contains("ya", na=False)
)


mask_qargidali = (df["category_sub"]=="Others") & (
    (name_lc.str.contains("qar", na=False) | name_lc.str.contains("qargid", na=False)) &
    name_lc.str.contains("ya", na=False)
)


mask_gunebaxan = (df["category_sub"]=="Others") & (
    (name_lc.str.contains("gunebaxan", na=False) | name_lc.str.contains("günəbaxan", na=False)) &
    name_lc.str.contains("ya", na=False)
)


df.loc[mask_kere, "category_sub"] = "Kərə yağı"
df.loc[mask_qargidali, "category_sub"] = "Qarğıdalı yağı"
df.loc[mask_gunebaxan, "category_sub"] = "Günəbaxan yağı"

df.loc[df["category_sub"].isin(["Kərə yağı","Qarğıdalı yağı","Günəbaxan yağı"]), "category_group"] = "Cooking fats"

print("Others count:", (df["category_sub"]=="Others").sum())

Others count: 126


In [ ]:
import numpy as np
import pandas as pd


FAT_FIX_RULES = [
    ("Günəbaxan yağı", r"g[üu]n[əe]baxan.*ya[gğ][ıi]"),
    ("Zeytun yağı",    r"zeytun.*ya[gğ][ıi]"),
    ("Qarğıdalı yağı", r"(qar[ğı]dal[ıi].*ya[gğ][ıi]|misir.*ya[gğ][ıi])"),
]


if "name_lc" not in df.columns:
    df["name_lc"] = df["product_name_raw"].astype("string").str.lower()

for sub, pattern in FAT_FIX_RULES:
    mask = (df["category_sub"] == "Others") & df["name_lc"].str.contains(pattern, regex=True, na=False)
    df.loc[mask, "category_sub"] = sub

GROUP_MAP_FIX = {
    "Günəbaxan yağı": "Cooking fats",
    "Zeytun yağı": "Cooking fats",
    "Qarğıdalı yağı": "Cooking fats",
}
df.loc[df["category_sub"].isin(GROUP_MAP_FIX.keys()), "category_group"] = df["category_sub"].map(GROUP_MAP_FIX)

print("Others count after fats fix:", (df["category_sub"] == "Others").sum())
df.loc[df["category_sub"] == "Others", "product_name_raw"].head(30)

Others count after fats fix: 114


C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\2451582572.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = (df["category_sub"] == "Others") & df["name_lc"].str.contains(pattern, regex=True, na=False)


4              Galina Blanca Göbələk 10qr
5     Ilgim Sarıkök Salafan Paket 15*40qr
6                 Limon Tozu 15qr Soyigit
7        Nid Uyudulmuş Sarımsaq Tozu 15qr
8              Nid Şokolad Damlaları 20qr
9                  Nid Xmeli Suneli 25 Gr
10             Acı Pul Bibər 15qr Soyigit
11     Acı Toz Qırmızı Bibər 15qr Soyigit
12          Nid Şəkərli Muncuq Paket 10qr
14                   Ilğım Pul Bibər 50qr
15      Misir Nishastasi 12x200qr Soyigit
16                 Nid Dəfnə Yarpağı 10qr
17              Nid Kartof Nisastasi 70qr
18                     Nid Quru Nane 50qr
20                      Ilgim Zirinc 50qr
21                     Nid Kəklikotu 50qr
22                     Kent Brinqer Vanil
23                Kent Kabartma Tozu 10qr
24                Ilgim Şəkərli Vanil 5qr
26               Ilgim Qabartma Tozu 10qr
27                    Kent Ət Bulyon 10qr
28                        Şəkərli Vanilin
29          Hamur Qabartma Tozu 5-li 50qr
30                             Sod

In [77]:
indices_to_drop=[4,5,6,7,8,9,10,11,12,15,16,17,18,20 ,21,22,23,24,26,27,29,30,31 ]
df=df.drop(indices_to_drop)

In [105]:
indices_to_drop=[156,157,158,159,160,161,162,163,164,165]
df=df.drop(indices_to_drop)

In [107]:
indices_to_drop=[171,177,185,186,187,188]
df=df.drop(indices_to_drop)

In [111]:
indices_to_drop=[88,194,195,198,199,203,204,205,215,217]
df=df.drop(indices_to_drop)

In [112]:
others_df = df[df["category_sub"] == "Others"]
display(others_df)

,scrape_run_id,scrape_ts_utc,snapshot_date,market,category_raw,product_name_raw,barcode,price_current_azn,price_old_azn,price_current_raw,...,unit_price_1l,unit_price_raw,unit_price_base,unit_price_base_label,name_lc,category_sub,category_group,barcode_str,cat_lc,category_raw_lc
87,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:04.992641,2026-02-13,Araz,makaronduyu,Ankara Vitaminli Fiyonk 500 qr,8.690577e+12,1.88,1.88,188,...,NaN,0.003760,3.760000,AZN / kg,ankara vitaminli fiyonk 500 qr,Others,Others,8690576529221,makaronduyu,makaronduyu
89,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:04.992641,2026-02-13,Araz,makaronduyu,Ankara Vitaminli Spaqetti 500 qr,8.690577e+12,1.88,1.88,188,...,NaN,0.003760,3.760000,AZN / kg,ankara vitaminli spaqetti 500 qr,Others,Others,8690576529009,makaronduyu,makaronduyu
190,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.095449,2026-02-13,Araz,sudmehsul,Milla Qaymaqlı Kənd Qatığı 6% 450 qr,4.760082e+12,2.38,2.38,238,...,NaN,0.005289,5.288889,AZN / kg,milla qaymaqlı kənd qatığı 6% 450 qr,Others,Others,4760081603830,sudmehsul,sudmehsul
193,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.095449,2026-02-13,Araz,sudmehsul,Burenkin Luq Göbelek Pendir 55% 100 qr,2.000000e+12,0.85,1.15,85,...,NaN,0.008500,8.500000,AZN / kg,burenkin luq göbelek pendir 55% 100 qr,Others,Others,2000000015729,sudmehsul,sudmehsul
200,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.127228,2026-02-13,Araz,temizlik,Bos Ağardıcı Persol 300qr,4.600905e+12,3.19,3.55,319,...,NaN,0.010633,10.633333,AZN / kg,bos ağardıcı persol 300qr,Others,Others,4600905000240,temizlik,temizlik
201,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.127228,2026-02-13,Araz,temizlik,Ağartıcı Persol Alafran 200qr,NaN,0.85,1.00,85,...,NaN,0.004250,4.250000,AZN / kg,ağartıcı persol alafran 200qr,Others,Others,NaN,temizlik,temizlik
202,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.127228,2026-02-13,Araz,temizlik,Ace Limon 1l,8.001480e+12,2.95,2.95,295,...,2.950000,0.002950,2.950000,AZN / L,ace limon 1l,Others,Others,8001480021891,temizlik,temizlik
208,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.127228,2026-02-13,Araz,temizlik,Ağardıcı Bloom Limon Təravətli 1 l,4.760091e+12,1.15,1.15,115,...,1.150000,0.001150,1.150000,AZN / L,ağardıcı bloom limon təravətli 1 l,Others,Others,4760090600752,temizlik,temizlik
214,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.127228,2026-02-13,Araz,temizlik,Bloom Şüşə Təmizləyən 500 ml,4.760091e+12,1.09,1.50,109,...,2.180000,0.002180,2.180000,AZN / L,bloom şüşə təmizləyən 500 ml,Others,Others,4760090600806,temizlik,temizlik
219,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.127228,2026-02-13,Araz,temizlik,Calgon Qoruyucu Toz Auto 500 qr,8.690571e+12,8.95,8.95,895,...,NaN,0.017900,17.900000,AZN / kg,calgon qoruyucu toz auto 500 qr,Others,Others,8690570530353,temizlik,temizlik


In [91]:
print("unique category_raw (top 30):")
print(df["category_raw"].astype("string").str.lower().str.strip().value_counts().head(30))

print("\nname_lc column exists?", "name_lc" in df.columns)
print("\nOthers count:", (df["category_sub"]=="Others").sum())

unique category_raw (top 30):
category_raw
sudmehsul      50
temizlik       50
yaglar         50
makaronduyu    46
seker          46
un             40
edviyat        13
Name: count, dtype: int64[pyarrow]

name_lc column exists? True

Others count: 74


In [ ]:
df["name_lc"] = df["product_name_raw"].astype("string").str.lower()
df["category_raw_lc"] = df["category_raw"].astype("string").str.lower().str.strip()

md = df["category_raw_lc"].str.contains(r"makaron.*duyu", na=False)

print("makaron*duyu rows:", md.sum())

rice_mask  = md & df["name_lc"].str.contains(r"düyü|duyu|basmati|sella", na=False)
pasta_mask = md & df["name_lc"].str.contains(r"makaron|əriştə|eriste|spagetti|penne|fusilli|vermi[şs]el", na=False)

df.loc[rice_mask,  "category_sub"] = "Düyü"
df.loc[pasta_mask, "category_sub"] = "Makaron"

df.loc[df["category_sub"].isin(["Düyü","Makaron","Un"]), "category_group"] = "Core staples"

print("Others count:", (df["category_sub"]=="Others").sum())

makaron*duyu rows: 46
Others count: 64


In [113]:
mask_un = df["category_raw"].astype("string").str.lower().str.strip().eq("un")

df.loc[mask_un, "category_sub"] = "Un"
df.loc[mask_un, "category_group"] = "Core staples"

print("Un count:", (df["category_sub"]=="Un").sum())
print("Others count:", (df["category_sub"]=="Others").sum())

Un count: 43
Others count: 18


In [124]:
others_df = df[df["category_sub"] == "Others"]
display(others_df)

,scrape_run_id,scrape_ts_utc,snapshot_date,market,category_raw,product_name_raw,barcode,price_current_azn,price_old_azn,price_current_raw,...,unit_price_1l,unit_price_raw,unit_price_base,unit_price_base_label,name_lc,category_sub,category_group,barcode_str,cat_lc,category_raw_lc
87,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:04.992641,2026-02-13,Araz,makaronduyu,Ankara Vitaminli Fiyonk 500 qr,8.690577e+12,1.88,1.88,188,...,NaN,0.003760,3.760000,AZN / kg,ankara vitaminli fiyonk 500 qr,Others,Others,8690576529221,makaronduyu,makaronduyu
89,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:04.992641,2026-02-13,Araz,makaronduyu,Ankara Vitaminli Spaqetti 500 qr,8.690577e+12,1.88,1.88,188,...,NaN,0.003760,3.760000,AZN / kg,ankara vitaminli spaqetti 500 qr,Others,Others,8690576529009,makaronduyu,makaronduyu
190,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.095449,2026-02-13,Araz,sudmehsul,Milla Qaymaqlı Kənd Qatığı 6% 450 qr,4.760082e+12,2.38,2.38,238,...,NaN,0.005289,5.288889,AZN / kg,milla qaymaqlı kənd qatığı 6% 450 qr,Others,Others,4760081603830,sudmehsul,sudmehsul


In [119]:
indicates_to_drop=[193]
df=df.drop(indicates_to_drop)

In [ ]:
cr = df["category_raw"].astype("string").str.lower().str.strip()
name = df["product_name_raw"].astype("string").str.lower()

mask_tem = cr.eq("temizlik")


mask_maye = mask_tem & name.str.contains(r"\b(\d+(\.\d+)?\s*(ml|l|lt))\b", regex=True, na=False)

mask_toz  = mask_tem & name.str.contains(r"\b(\d+(\.\d+)?\s*(kq|kg|qr))\b", regex=True, na=False)


df.loc[mask_maye, "category_sub"] = "Yuyucu maye"
df.loc[mask_toz,  "category_sub"] = "Yuyucu toz"


df.loc[mask_tem & (df["category_sub"].isin(["Yuyucu maye", "Yuyucu toz"])), "category_group"] = "Non food"

print("temizlik total:", mask_tem.sum())
print("maye:", mask_maye.sum(), "toz:", mask_toz.sum())

temizlik total: 37
maye: 22 toz: 15


C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\2738795926.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_maye = mask_tem & name.str.contains(r"\b(\d+(\.\d+)?\s*(ml|l|lt))\b", regex=True, na=False)
C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\2738795926.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_toz  = mask_tem & name.str.contains(r"\b(\d+(\.\d+)?\s*(kq|kg|qr))\b", regex=True, na=False)


In [126]:
df.columns

Index(['scrape_run_id', 'scrape_ts_utc', 'snapshot_date', 'market',
       'category_raw', 'product_name_raw', 'barcode', 'price_current_azn',
       'price_old_azn', 'price_current_raw', 'price_old_raw', 'promo_flag',
       'purchasable_balance', 'availability_status', 'unit_info_raw',
       'product_hash', 'dq_missing_barcode', 'dq_invalid_price',
       'dq_missing_purchasable_balance', 'scrape_ts', 'scrape_ts_baku',
       'baku_hour', 'session', 'market_day', 'market_day_session',
       'std_qty_total', 'std_unit', 'pack_multiplier', 'dq_missing_unit',
       'unit_price_1kg', 'unit_price_1l', 'unit_price_raw', 'unit_price_base',
       'unit_price_base_label', 'name_lc', 'category_sub', 'category_group',
       'barcode_str', 'cat_lc', 'category_raw_lc'],
      dtype='str')

In [128]:
df['cat_lc']

0      edviyat
1      edviyat
2      edviyat
3      edviyat
13     edviyat
        ...   
331     yaglar
332     yaglar
333     yaglar
334     yaglar
335     yaglar
Name: cat_lc, Length: 248, dtype: string

In [ ]:
cr = df["category_raw"].astype("string").str.lower().str.strip()
name = df["product_name_raw"].astype("string").str.lower()

mask_other = df["category_sub"].eq("Others")

m = mask_other & cr.eq("makaronduyu")
df.loc[m, "category_sub"] = "Makaron"
df.loc[m, "category_group"] = "Core staples"

m = mask_other & cr.eq("sudmehsul")

df.loc[m & name.str.contains(r"(qatıq|qatiq)", na=False), "category_sub"] = "Qatıq"

df.loc[m & df["category_sub"].eq("Others"), "category_sub"] = "Dairy (Other)"

df.loc[m, "category_group"] = "Dairy"

print("Others left:", (df["category_sub"]=="Others").sum())

Others left: 0


C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\3182985194.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df.loc[m & name.str.contains(r"(qatıq|qatiq)", na=False), "category_sub"] = "Qatıq"


In [130]:
df.columns

Index(['scrape_run_id', 'scrape_ts_utc', 'snapshot_date', 'market',
       'category_raw', 'product_name_raw', 'barcode', 'price_current_azn',
       'price_old_azn', 'price_current_raw', 'price_old_raw', 'promo_flag',
       'purchasable_balance', 'availability_status', 'unit_info_raw',
       'product_hash', 'dq_missing_barcode', 'dq_invalid_price',
       'dq_missing_purchasable_balance', 'scrape_ts', 'scrape_ts_baku',
       'baku_hour', 'session', 'market_day', 'market_day_session',
       'std_qty_total', 'std_unit', 'pack_multiplier', 'dq_missing_unit',
       'unit_price_1kg', 'unit_price_1l', 'unit_price_raw', 'unit_price_base',
       'unit_price_base_label', 'name_lc', 'category_sub', 'category_group',
       'barcode_str', 'cat_lc', 'category_raw_lc'],
      dtype='str')

In [136]:
df[['category_sub', 'category_group']]

,category_sub,category_group
0,Duz,Everyday essentials
1,Duz,Everyday essentials
2,Duz,Everyday essentials
3,Duz,Everyday essentials
13,Duz,Everyday essentials
...,...,...
331,Günəbaxan yağı,Cooking fats
332,Günəbaxan yağı,Cooking fats
333,Günəbaxan yağı,Cooking fats
334,Günəbaxan yağı,Cooking fats


In [137]:
df['category_sub'].unique()

<ArrowStringArray>
[           'Duz',        'Makaron',           'Düyü',             'Un',
          'Şəkər',            'Süd',          'Qatıq',  'Dairy (Other)',
     'Yuyucu toz',    'Yuyucu maye', 'Qarğıdalı yağı', 'Günəbaxan yağı',
    'Zeytun yağı',      'Kərə yağı']
Length: 14, dtype: str

In [142]:
df[df["category_sub"]=="Dairy (Other)"][["product_name_raw"]].head(20)

,product_name_raw
190,Milla Qaymaqlı Kənd Qatığı 6% 450 qr


In [143]:
df["category_sub"] = df["category_sub"].replace("Dairy (Other)", "Qatıq")

In [144]:
df['category_sub'].unique()

<ArrowStringArray>
[           'Duz',        'Makaron',           'Düyü',             'Un',
          'Şəkər',            'Süd',          'Qatıq',     'Yuyucu toz',
    'Yuyucu maye', 'Qarğıdalı yağı', 'Günəbaxan yağı',    'Zeytun yağı',
      'Kərə yağı']
Length: 13, dtype: str

In [147]:
df[df["name_lc"].str.contains(r"\bayran\b", regex=True, na=False)][
    ["product_name_raw","category_raw","category_sub"]
].head(50)

,product_name_raw,category_raw,category_sub


In [148]:
df.columns

Index(['scrape_run_id', 'scrape_ts_utc', 'snapshot_date', 'market',
       'category_raw', 'product_name_raw', 'barcode', 'price_current_azn',
       'price_old_azn', 'price_current_raw', 'price_old_raw', 'promo_flag',
       'purchasable_balance', 'availability_status', 'unit_info_raw',
       'product_hash', 'dq_missing_barcode', 'dq_invalid_price',
       'dq_missing_purchasable_balance', 'scrape_ts', 'scrape_ts_baku',
       'baku_hour', 'session', 'market_day', 'market_day_session',
       'std_qty_total', 'std_unit', 'pack_multiplier', 'dq_missing_unit',
       'unit_price_1kg', 'unit_price_1l', 'unit_price_raw', 'unit_price_base',
       'unit_price_base_label', 'name_lc', 'category_sub', 'category_group',
       'barcode_str', 'cat_lc', 'category_raw_lc'],
      dtype='str')

In [156]:
df['product_name_raw'].unique()

<ArrowStringArray>
[   'Mastero Xörək Duzu Yodlaşdırılmış 1000 qr',
                   'Original Xörək Duzu 900 qr',
          'Bizim Duz Xörək Duzu Plastik 850 qr',
                         'Tac Narin Duz 650 qr',
                          'Nid Limon Duzu 90qr',
             'Atlantika Dəniz Duzu Yodlu 500qr',
                          'Azerduz Paket 750gr',
                       'Salute Xörək Duzu 1 Kq',
              'Azərduz Süfrə Duzu Paket 1.5 kq',
                        'Tac Xörək Duzu 900 qr',
 ...
                   'Final Qarğıdalı Yağı 0.5 l',
            'Krestyanskoe Eridilmiş Yağ 800 qr',
                    'Komili Qarğıdalı Yağı 2 l',
          'Consul Zeytun Yağı (Şüşə Qabda) 1 l',
 'Kirlangic Günəbaxan Yağı 2 l (Plastik Qabda)',
            'Saratovskoye Günəbaxan Yağı 0.9 L',
                    'Yurdum Günəbaxan Yağı 1lt',
                   'Final Günəbaxan Yağı 0.5lt',
            'Saratovskoye Günəbaxan Yağı 4.5 L',
         'Consul 2lt Zeytun Yağı (Dəmir Qabda

In [157]:
with open("product_name_raw.txt", "w", encoding="utf-8") as f:
    for name in df['product_name_raw'].astype(str):
        f.write(name + "\n")


In [158]:
df.columns

Index(['scrape_run_id', 'scrape_ts_utc', 'snapshot_date', 'market',
       'category_raw', 'product_name_raw', 'barcode', 'price_current_azn',
       'price_old_azn', 'price_current_raw', 'price_old_raw', 'promo_flag',
       'purchasable_balance', 'availability_status', 'unit_info_raw',
       'product_hash', 'dq_missing_barcode', 'dq_invalid_price',
       'dq_missing_purchasable_balance', 'scrape_ts', 'scrape_ts_baku',
       'baku_hour', 'session', 'market_day', 'market_day_session',
       'std_qty_total', 'std_unit', 'pack_multiplier', 'dq_missing_unit',
       'unit_price_1kg', 'unit_price_1l', 'unit_price_raw', 'unit_price_base',
       'unit_price_base_label', 'name_lc', 'category_sub', 'category_group',
       'barcode_str', 'cat_lc', 'category_raw_lc'],
      dtype='str')

In [160]:
df['std_qty_total']

0      1000.0
1       900.0
2       850.0
3       650.0
13       90.0
        ...  
331     900.0
332    1000.0
333     500.0
334    4500.0
335    2000.0
Name: std_qty_total, Length: 248, dtype: float64

In [ ]:
import re
import numpy as np
import pandas as pd

BRANDS = [
    "Bizim duz", "Tac", "Nid", "Atlantika", "Azərduz", "Salute",
    "Ankara", "Barilla", "Mastero", "Original", "Doymak", "Tamaşa",
    "Saville", "Bismak", "Oman", "Makfa", "Sultan", "Mohobbat",
    "Azər şəkər", "Ləzzət", "Can", "Milla", "İvanovka", "Pinar",
    "Consul", "Saratovskoye", "Final", "Yurdum", "Kirlangic", "Komili",
    "Krestyanskoe", "Ana bala", "Möcüzə", "Zeytun bağları", "Super Sun",
    "Zolotoe solnışko", "Zateya", "Avedov", "Borges", "Sunar", "Miad",
    "Taris Rivera", "Dr. Oetker", "Qudvill", "Car", "Favelli", "Rollton",
    "Endaksi", "Pudov", "Anadolu", "Petrovskiye nivi", "Soba", "Kent",
    "Bloom", "Crystalex", "Ace", "Domestos", "Ariel", "Cilit Bang",
    "Vanish", "Calgon", "Perwoll", "Persol", "Belarus moloçniy mir",
    "Richmond", "İmişli", "Dişdəmə", "Seçmə", "Qidam", "Atena",
    "Kehriz", "Prostokvashino", "Port De Kalleh",
    "Dr. Milk", "Doktor Milk"
]

def _norm_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip())

BRANDS = [_norm_spaces(b) for b in BRANDS]

ALIASES = {
    
    r"\b(dr\.?\s*milk|doktor\s*milk)\b": "Dr. Milk",
    r"\b(age|ace)\b": "Ace",
    r"\bzeytun\s*ba[ğg]lar[ıi]\b": "Zeytun bağları",
    r"\bimisli\b": "İmişli",
}


brands_sorted = sorted(BRANDS, key=len, reverse=True)
brand_pattern = r"(?i)\b(" + "|".join(re.escape(b) for b in brands_sorted) + r")\b"

name = df["product_name_raw"].astype("string").fillna("").map(_norm_spaces)

name_lc = name.str.lower()

df["brand"] = pd.NA

for pat, canon in ALIASES.items():
    m = name_lc.str.contains(pat, regex=True, na=False)
    df.loc[m & df["brand"].isna(), "brand"] = canon

m2 = name.str.extract(brand_pattern, expand=False)
df.loc[df["brand"].isna(), "brand"] = m2

df["brand"] = df["brand"].astype("string")
df.loc[df["brand"].isna() | (df["brand"].str.len() == 0), "brand"] = pd.NA


print("Brand tapılmayan (NaN) say:", df["brand"].isna().sum())
df[["product_name_raw", "brand"]].head(20)

Brand tapılmayan (NaN) say: 20


C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\2068991251.py:55: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m = name_lc.str.contains(pat, regex=True, na=False)
C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_16248\2068991251.py:55: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m = name_lc.str.contains(pat, regex=True, na=False)


,product_name_raw,brand
0,Mastero Xörək Duzu Yodlaşdırılmış 1000 qr,Mastero
1,Original Xörək Duzu 900 qr,Original
2,Bizim Duz Xörək Duzu Plastik 850 qr,Bizim Duz
3,Tac Narin Duz 650 qr,Tac
13,Nid Limon Duzu 90qr,Nid
19,Atlantika Dəniz Duzu Yodlu 500qr,Atlantika
25,Azerduz Paket 750gr,<NA>
40,Salute Xörək Duzu 1 Kq,Salute
41,Azərduz Süfrə Duzu Paket 1.5 kq,Azərduz
46,Tac Xörək Duzu 900 qr,Tac


In [162]:
df['brand'].isnull().sum()

np.int64(20)

In [ ]:

nan_brands = df[df['brand'].isna()]
display(nan_brands)

,scrape_run_id,scrape_ts_utc,snapshot_date,market,category_raw,product_name_raw,barcode,price_current_azn,price_old_azn,price_current_raw,...,unit_price_base,unit_price_base_label,name_lc,category_sub,category_group,barcode_str,cat_lc,category_raw_lc,catraw_lc,brand
25,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:04.946584,2026-02-13,Araz,edviyat,Azerduz Paket 750gr,4.760073e+12,0.59,0.59,59,...,0.786667,AZN / kg,azerduz paket 750gr,Duz,Everyday essentials,4760072900016,edviyat,edviyat,edviyat,<NA>
103,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.024337,2026-02-13,Araz,seker,Azerşəkər Kəllə Qənd 800qr,4.760093e+12,2.55,2.55,255,...,3.187500,AZN / kg,azerşəkər kəllə qənd 800qr,Şəkər,Everyday essentials,4760093400038,seker,seker,seker,<NA>
113,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.024337,2026-02-13,Araz,seker,Azərşəkər Şəkər Tozu 5kg,4.760057e+12,10.00,10.00,1000,...,2.000000,AZN / kg,azərşəkər şəkər tozu 5kg,Şəkər,Everyday essentials,4760056701134,seker,seker,seker,<NA>
116,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.024337,2026-02-13,Araz,seker,Secme Şəkər Tozu Paket 1500qr,4.760151e+12,3.55,3.55,355,...,2.366667,AZN / kg,secme şəkər tozu paket 1500qr,Şəkər,Everyday essentials,4760150702327,seker,seker,seker,<NA>
129,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.024337,2026-02-13,Araz,seker,Kəhriz Nanəli Noğul Paket 500qr,4.760126e+12,2.30,2.30,230,...,4.600000,AZN / kg,kəhriz nanəli noğul paket 500qr,Şəkər,Everyday essentials,4760125800256,seker,seker,seker,<NA>
288,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.193487,2026-02-13,Araz,yaglar,Kırlangıç Qarğıdali Yağı 1l,8.697420e+12,5.99,7.40,599,...,5.990000,AZN / L,kırlangıç qarğıdali yağı 1l,Others,Cooking fats,8697419812533,yaglar,yaglar,yaglar,<NA>
289,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.193487,2026-02-13,Araz,yaglar,Kırlangıç Günəbaxan Yağı 1l,8.697420e+12,6.30,6.30,630,...,6.300000,AZN / L,kırlangıç günəbaxan yağı 1l,Others,Cooking fats,8697419812038,yaglar,yaglar,yaglar,<NA>
303,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.193487,2026-02-13,Araz,yaglar,Mocuze Qargidali Yagi 1l,4.760061e+12,7.35,7.35,735,...,7.350000,AZN / L,mocuze qargidali yagi 1l,Others,Cooking fats,4760060600218,yaglar,yaglar,yaglar,<NA>
307,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.193487,2026-02-13,Araz,yaglar,Zeytun Baıları Zeytun Yağı 750ml Pomace,4.760061e+12,15.50,15.50,1550,...,20.666667,AZN / L,zeytun baıları zeytun yağı 750ml pomace,Others,Cooking fats,4760060600577,yaglar,yaglar,yaglar,<NA>


In [170]:
indicates_to_drop=[167]
df=df.drop(indicates_to_drop)

In [172]:
df.columns

Index(['scrape_run_id', 'scrape_ts_utc', 'snapshot_date', 'market',
       'category_raw', 'product_name_raw', 'barcode', 'price_current_azn',
       'price_old_azn', 'price_current_raw', 'price_old_raw', 'promo_flag',
       'purchasable_balance', 'availability_status', 'unit_info_raw',
       'product_hash', 'dq_missing_barcode', 'dq_invalid_price',
       'dq_missing_purchasable_balance', 'scrape_ts', 'scrape_ts_baku',
       'baku_hour', 'session', 'market_day', 'market_day_session',
       'std_qty_total', 'std_unit', 'pack_multiplier', 'dq_missing_unit',
       'unit_price_1kg', 'unit_price_1l', 'unit_price_raw', 'unit_price_base',
       'unit_price_base_label', 'name_lc', 'category_sub', 'category_group',
       'barcode_str', 'cat_lc', 'category_raw_lc', 'catraw_lc', 'brand'],
      dtype='str')

In [ ]:
import re
import pandas as pd

AZ_MAP = str.maketrans({
    "ə":"e","Ə":"e",
    "ş":"s","Ş":"s",
    "ı":"i","İ":"i",
    "ğ":"g","Ğ":"g",
    "ç":"c","Ç":"c",
    "ö":"o","Ö":"o",
    "ü":"u","Ü":"u",
})

def to_ascii(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.translate(AZ_MAP)
    s = re.sub(r"\s+", " ", s.strip())
    return s.lower()



ALIASES = {
    
    r"\bazerduz\b": "Azərduz",

    r"\bazer\s*seker\b": "Azər şəkər",
    r"\bazerseker\b": "Azər şəkər",

    r"\bsecme\s*seker\b": "Seçmə",

    r"\bkehriz\b": "Kehriz",

    r"\bkirlangic\b": "Kirlangic",

    r"\bmocuze\b": "Möcüzə",

    r"\bzeytun\s*baglari\b": "Zeytun bağları",
    r"\bzeytun\s*bailari\b": "Zeytun bağları",
}

In [ ]:
name_raw = df["product_name_raw"].astype("string").fillna("")
name_ascii = name_raw.map(to_ascii)

df["brand"] = pd.NA


for pat, canon in ALIASES.items():
    m = name_ascii.str.contains(pat, regex=True, na=False)
    df.loc[m & df["brand"].isna(), "brand"] = canon

brands_sorted = sorted(BRANDS, key=len, reverse=True)
brands_ascii = [to_ascii(b) for b in brands_sorted]

brand_pattern_ascii = r"\b(" + "|".join(re.escape(b) for b in brands_ascii) + r")\b"
m2 = name_ascii.str.extract(brand_pattern_ascii, expand=False)

ascii_to_canon = {to_ascii(b): b for b in BRANDS}
df.loc[df["brand"].isna(), "brand"] = m2.map(ascii_to_canon)

print("Brand tapılmayan (NaN) say:", df["brand"].isna().sum())

Brand tapılmayan (NaN) say: 1


In [178]:
df['brand'].isnull().sum()

np.int64(1)

In [179]:
nan_brands = df[df['brand'].isna()]
display(nan_brands)

,scrape_run_id,scrape_ts_utc,snapshot_date,market,category_raw,product_name_raw,barcode,price_current_azn,price_old_azn,price_current_raw,...,unit_price_base,unit_price_base_label,name_lc,category_sub,category_group,barcode_str,cat_lc,category_raw_lc,catraw_lc,brand
243,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:05.127228,2026-02-13,Araz,temizlik,Age Qab Yuyan Sf 12* 1 Lt,4.760091e+12,1.35,1.35,135,...,0.1125,AZN / L,age qab yuyan sf 12* 1 lt,Yuyucu maye,Non food,4760090601179,temizlik,temizlik,temizlik,NaN


In [180]:
df = df.dropna(subset=['brand'])

In [181]:
df.columns

Index(['scrape_run_id', 'scrape_ts_utc', 'snapshot_date', 'market',
       'category_raw', 'product_name_raw', 'barcode', 'price_current_azn',
       'price_old_azn', 'price_current_raw', 'price_old_raw', 'promo_flag',
       'purchasable_balance', 'availability_status', 'unit_info_raw',
       'product_hash', 'dq_missing_barcode', 'dq_invalid_price',
       'dq_missing_purchasable_balance', 'scrape_ts', 'scrape_ts_baku',
       'baku_hour', 'session', 'market_day', 'market_day_session',
       'std_qty_total', 'std_unit', 'pack_multiplier', 'dq_missing_unit',
       'unit_price_1kg', 'unit_price_1l', 'unit_price_raw', 'unit_price_base',
       'unit_price_base_label', 'name_lc', 'category_sub', 'category_group',
       'barcode_str', 'cat_lc', 'category_raw_lc', 'catraw_lc', 'brand'],
      dtype='str')

In [ ]:
import pandas as pd
import numpy as np
import re
import hashlib

def get_col(df, col):
    if col in df.columns:
        return df[col]
    return pd.Series([pd.NA] * len(df), index=df.index, dtype="string")


def clean_barcode_series(s: pd.Series) -> pd.Series:
    s = s.astype("string")
    s = (
        s.str.strip()
         .str.replace(r"\.0$", "", regex=True)
         .str.replace(r"\D", "", regex=True)
    )
    s = s.replace(["", "None", "nan", "<NA>"], pd.NA)
    return s

def is_valid_ean(code) -> bool:
    if pd.isna(code):
        return False
    code = str(code)
    if len(code) not in (8, 12, 13, 14):
        return False
    if not code.isdigit():
        return False

    digits = [int(c) for c in code]
    check = digits[-1]
    body = digits[:-1]

    body_rev = body[::-1]
    total = 0
    for i, d in enumerate(body_rev, start=1):
        total += d * 3 if i % 2 == 1 else d 
    calc = (10 - (total % 10)) % 10
    return calc == check

df["barcode_clean"] = clean_barcode_series(get_col(df, "barcode"))
df["valid_barcode"] = df["barcode_clean"].apply(is_valid_ean)

df["dq_missing_barcode"] = df["barcode_clean"].isna().astype(int)
df["dq_invalid_barcode"] = (df["barcode_clean"].notna() & ~df["valid_barcode"]).astype(int)

df.loc[~df["valid_barcode"], "barcode_clean"] = pd.NA


df["sku_str"] = clean_barcode_series(get_col(df, "sku"))  
df["global_product_id_sku"] = pd.NA

if "sku_bridge" in globals() and isinstance(sku_bridge, pd.DataFrame):
    
    if set(["market","sku","global_product_id"]).issubset(sku_bridge.columns):
        tmp = sku_bridge.copy()
        tmp["sku"] = tmp["sku"].astype("string").str.strip()

        df = df.merge(
            tmp[["market","sku","global_product_id"]],
            how="left",
            left_on=["market", "sku_str"],
            right_on=["market", "sku"]
        )
        df["global_product_id_sku"] = df["global_product_id"]
        df = df.drop(columns=["sku", "global_product_id"], errors="ignore")


qty = get_col(df, "net_qty").astype("Float64")
unit = get_col(df, "unit").astype("string").str.lower().str.strip()

df["std_qty"] = qty  
df.loc[unit.isin(["l", "lt", "litr", "liter"]), "std_qty"] = qty * 1000
df.loc[unit.isin(["kg", "kq"]), "std_qty"] = qty * 1000

def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9 ]", "", s)
    return s.strip()

name_clean = get_col(df, "product_name").fillna(get_col(df, "product_name_raw")).astype("string")
df["name_norm2"] = name_clean.apply(normalize_text)


std_qty_str = df["std_qty"].astype("Float64").astype("string").replace(["<NA>", "nan", "None"], "")

df["identity_string"] = df["name_norm2"] + "|" + std_qty_str

df["identity_hash"] = df["identity_string"].apply(
    lambda x: hashlib.sha256(str(x).encode("utf-8")).hexdigest()
)


df["global_product_id"] = np.select(
    [
        df["barcode_clean"].notna(),
        df["global_product_id_sku"].notna()
    ],
    [
        df["barcode_clean"],           
        df["global_product_id_sku"]    
    ],
    default=df["identity_hash"]        
)

df["dq_missing_gpid"] = df["global_product_id"].isna().astype(int)


df["gpid_hash"] = df["global_product_id"].apply(lambda s: hashlib.sha256(str(s).encode("utf-8")).hexdigest())

In [183]:
df.columns

Index(['scrape_run_id', 'scrape_ts_utc', 'snapshot_date', 'market',
       'category_raw', 'product_name_raw', 'barcode', 'price_current_azn',
       'price_old_azn', 'price_current_raw', 'price_old_raw', 'promo_flag',
       'purchasable_balance', 'availability_status', 'unit_info_raw',
       'product_hash', 'dq_missing_barcode', 'dq_invalid_price',
       'dq_missing_purchasable_balance', 'scrape_ts', 'scrape_ts_baku',
       'baku_hour', 'session', 'market_day', 'market_day_session',
       'std_qty_total', 'std_unit', 'pack_multiplier', 'dq_missing_unit',
       'unit_price_1kg', 'unit_price_1l', 'unit_price_raw', 'unit_price_base',
       'unit_price_base_label', 'name_lc', 'category_sub', 'category_group',
       'barcode_str', 'cat_lc', 'category_raw_lc', 'catraw_lc', 'brand',
       'barcode_clean', 'valid_barcode', 'dq_invalid_barcode', 'sku_str',
       'global_product_id_sku', 'std_qty', 'name_norm2', 'identity_string',
       'identity_hash', 'global_product_id', 'dq_missi

In [ ]:
import pandas as pd
import numpy as np
import re
import hashlib


def get_col(df, col):
    if col in df.columns:
        return df[col]
    return pd.Series([pd.NA] * len(df), index=df.index, dtype="string")


def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9 ]", "", s)
    return s.strip()

name_raw = get_col(df, "product_name_raw").astype("string")
df["product_name_clean"] = name_raw.str.strip()


df["name_norm"] = df["product_name_clean"].apply(normalize_text)

df["product_canonical"] = df["name_norm"]


def parse_multipack(name):
    if pd.isna(name):
        return 1, None, 1
    
    name = str(name).lower()

    
    m = re.search(r"(\d+)\s*[x×]\s*(\d+)", name)
    if m:
        count = int(m.group(1))
        qty = int(m.group(2))
        return qty, None, count
    
    return 1, None, 1

df[["multi_qty","_tmp","multi_count"]] = (
    df["product_name_clean"]
    .apply(lambda x: pd.Series(parse_multipack(x)))
)

df["multi_unit"] = get_col(df, "std_unit")
df.drop(columns=["_tmp"], errors="ignore", inplace=True)


std_qty_str = (
    get_col(df, "std_qty_total")
    .astype("Float64")
    .astype("string")
    .replace(["<NA>","nan","None"], "")
)

brand = get_col(df, "brand").astype("string").fillna("")

df["identity_string_strong"] = (
    df["product_canonical"] + "|" +
    brand + "|" +
    std_qty_str + "|" +
    df["std_unit"].astype("string").fillna("")
)

df["identity_hash_strong"] = df["identity_string_strong"].apply(
    lambda x: hashlib.sha256(str(x).encode("utf-8")).hexdigest()
)


df["global_product_id"] = np.select(
    [
        df["barcode_clean"].notna(),
        df["global_product_id_sku"].notna()
    ],
    [
        df["barcode_clean"],
        df["global_product_id_sku"]
    ],
    default=df["identity_hash_strong"]
)

df["dq_missing_gpid"] = df["global_product_id"].isna().astype(int)

df["gpid_hash"] = df["global_product_id"].apply(
    lambda s: hashlib.sha256(str(s).encode("utf-8")).hexdigest()
)

In [8]:
df.columns

Index(['scrape_run_id', 'scrape_ts_utc', 'snapshot_date', 'market',
       'category_raw', 'product_name_raw', 'barcode', 'price_current_azn',
       'price_old_azn', 'price_current_raw', 'price_old_raw', 'promo_flag',
       'purchasable_balance', 'availability_status', 'unit_info_raw',
       'product_hash', 'dq_missing_barcode', 'dq_invalid_price',
       'dq_missing_purchasable_balance', 'scrape_ts', 'scrape_ts_baku',
       'baku_hour', 'session', 'market_day', 'market_day_session',
       'std_qty_total', 'std_unit', 'pack_multiplier', 'dq_missing_unit',
       'unit_price_1kg', 'unit_price_1l', 'unit_price_raw', 'unit_price_base',
       'unit_price_base_label', 'name_lc', 'category_sub', 'category_group',
       'barcode_str', 'cat_lc', 'category_raw_lc', 'catraw_lc', 'brand',
       'barcode_clean', 'valid_barcode', 'dq_invalid_barcode', 'sku_str',
       'global_product_id_sku', 'std_qty', 'name_norm2', 'identity_string',
       'identity_hash', 'global_product_id', 'dq_missi

In [ ]:
import pandas as pd
import numpy as np

print("======================================")
print("FINAL DATA QUALITY AUDIT STARTED")
print("======================================")

TOTAL_ROWS = len(df)

critical_cols = [
    "product_name_clean",
    "brand",
    "std_qty_total",
    "std_unit",
    "category_sub",
    "category_group",
    "global_product_id",
    "identity_hash_strong"
]

null_report = {}

for col in critical_cols:
    if col in df.columns:
        null_count = df[col].isna().sum()
        null_report[col] = null_count

print("\n--- Critical Null Report ---")
for k,v in null_report.items():
    print(f"{k}: {v} null ({round(v/TOTAL_ROWS*100,2)}%)")

FINAL DATA QUALITY AUDIT STARTED

--- Critical Null Report ---
product_name_clean: 0 null (0.0%)
brand: 0 null (0.0%)
std_qty_total: 0 null (0.0%)
std_unit: 0 null (0.0%)
category_sub: 0 null (0.0%)
category_group: 0 null (0.0%)
global_product_id: 0 null (0.0%)
identity_hash_strong: 0 null (0.0%)


In [ ]:

empty_report = {}

for col in critical_cols:
    if col in df.columns:
        empty_count = (df[col].astype("string").str.strip() == "").sum()
        empty_report[col] = empty_count

print("\n--- Empty String Report ---")
for k,v in empty_report.items():
    print(f"{k}: {v} empty")


--- Empty String Report ---
product_name_clean: 0 empty
brand: 0 empty
std_qty_total: 0 empty
std_unit: 0 empty
category_sub: 0 empty
category_group: 0 empty
global_product_id: 0 empty
identity_hash_strong: 0 empty


In [ ]:

dup_gpid = df["global_product_id"].duplicated().sum()

print("\n--- Duplicate GPID ---")
print(f"Duplicate GPID rows: {dup_gpid}")



--- Duplicate GPID ---
Duplicate GPID rows: 0


In [ ]:

barcode_conflict = df[
    (df["barcode_clean"].notna()) &
    (df["global_product_id"] != df["barcode_clean"])
]

print("\n--- Barcode Priority Conflict ---")
print(f"Conflicts: {len(barcode_conflict)}")


--- Barcode Priority Conflict ---
Conflicts: 0


In [ ]:

collision_check = (
    df.groupby("identity_hash_strong")["product_name_clean"]
    .nunique()
)

identity_collisions = (collision_check > 1).sum()

print("\n--- Identity Collision ---")
print(f"Strong identity collisions: {identity_collisions}")


--- Identity Collision ---
Strong identity collisions: 0


In [ ]:

if "price_current_azn" in df.columns:
    negative_price = (df["price_current_azn"] < 0).sum()
else:
    negative_price = 0

print("\n--- Price Sanity ---")
print(f"Negative prices: {negative_price}")


--- Price Sanity ---
Negative prices: 0


In [ ]:

total_issues = (
    sum(null_report.values()) +
    sum(empty_report.values()) +
    dup_gpid +
    len(barcode_conflict) +
    identity_collisions +
    negative_price
)

print("\n======================================")
if total_issues == 0:
    print("DATA STATUS: ✅ CLEAN - SAFE TO SAVE")
else:
    print("DATA STATUS: ⚠ CHECK WARNINGS BEFORE SAVE")
print("======================================")




DATA STATUS: ✅ CLEAN - SAFE TO SAVE


In [194]:
print("\n========== DQ FLAGS CONSISTENCY ==========")
dq_cols = [c for c in df.columns if c.startswith("dq_")]
for col in dq_cols:
    print(col, "=", df[col].sum())



========== DQ FLAGS CONSISTENCY ==========
dq_missing_barcode = 11
dq_invalid_price = 0
dq_missing_purchasable_balance = 0
dq_missing_unit = 7
dq_invalid_barcode = 0
dq_missing_gpid = 0


In [196]:
df[df["dq_missing_unit"] == 1][
    ["product_name_raw","std_qty_total","std_unit"]
]

,product_name_raw,std_qty_total,std_unit
262,Anadolu Un kq Çəki,1000.0,g
263,Tac Buğda Unu kq,1000.0,g
267,Tac Makaron Vermişel kq,1000.0,g
268,Tac Makaron Qələm kq,1000.0,g
269,Tac Makaron Burma kq,1000.0,g
270,Tac Makaron Spiral kq,1000.0,g
271,Tac Makaron İncə Boru kq,1000.0,g


In [198]:
df["dq_missing_unit"] = (
    df["std_qty_total"].isna() |
    df["std_unit"].isna()
).astype(int)

In [199]:
df["dq_missing_unit"].sum()

np.int64(0)

In [197]:
print("\n========== AVAILABILITY CHECK ==========")
if "purchasable_balance" in df.columns:
    print("Null purchasable_balance:",
          df["purchasable_balance"].isna().sum())
    print("Negative balance:",
          (df["purchasable_balance"] < 0).sum())

if "availability_status" in df.columns:
    print("Availability distribution:")
    print(df["availability_status"].value_counts())


========== AVAILABILITY CHECK ==========
Null purchasable_balance: 0
Negative balance: 0
Availability distribution:
availability_status
available       226
out_of_stock     10
Name: count, dtype: int64


In [195]:
print("\n========== PROMO CHECK ==========")
if "promo_flag" in df.columns:
    print("Promo distribution:")
    print(df["promo_flag"].value_counts())


========== PROMO CHECK ==========
Promo distribution:
promo_flag
0    159
1     77
Name: count, dtype: int64


In [200]:
save_path = "araz_clean_final.csv"

df.to_csv(save_path, index=False, encoding="utf-8-sig")

print(f"\nSaved to: {save_path}")
print("Rows:", len(df))


Saved to: araz_clean_final.csv
Rows: 236


In [ ]:
import os
from datetime import datetime

today = "2026_02_13"

silver_dir = "silver_layer"
os.makedirs(silver_dir, exist_ok=True)

save_path = f"{silver_dir}/araz_silver_{today}.csv"

df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Silver layer created:")
print(save_path)
print("Rows:", len(df))

Silver layer created:
silver_layer/araz_silver_2026_02_13.csv
Rows: 236


In [4]:
import pandas as pd
from pathlib import Path

Silver_file = Path("silver_layer")/"araz_silver_2026_02_13.csv"

df=pd.read_csv(Silver_file,encoding="utf-8-sig")

print(len(df))

df.head(1)

236


,scrape_run_id,scrape_ts_utc,snapshot_date,market,category_raw,product_name_raw,barcode,price_current_azn,price_old_azn,price_current_raw,...,dq_missing_gpid,gpid_hash,product_name_clean,name_norm,product_canonical,multi_qty,multi_count,multi_unit,identity_string_strong,identity_hash_strong
0,2013e872-127c-40d3-90ba-80f4d1f89926,2026-02-13 20:02:04.946584,2026-02-13,Araz,edviyat,Mastero Xörək Duzu Yodlaşdırılmış 1000 qr,4.760106e+12,0.7,0.7,70,...,0,5f0e212f8601dfcd371fac4e107659905cfd975e360192...,Mastero Xörək Duzu Yodlaşdırılmış 1000 qr,mastero xrk duzu yodladrlm 1000 qr,mastero xrk duzu yodladrlm 1000 qr,1.0,1.0,g,mastero xrk duzu yodladrlm 1000 qr|Mastero|100...,c0472bd4ec75b0d66aa36a44b3c030c9aec98dd7440a01...


In [ ]:
import re

s = df["product_name_raw"].astype("string")

mask_kq = s.str.contains(r"(\bkq\b|\bkg\b|\bkilo\b|kq\s*çəki)", case=False, na=False)

mask_anadolu = s.str.contains(r"\banadolu\b", case=False, na=False) & mask_kq
mask_tac     = s.str.contains(r"\btac\b",     case=False, na=False) & mask_kq

print("Anadolu kq std_qty_total null:", df.loc[mask_anadolu, "std_qty_total"].isna().sum())
print("Tac kq std_qty_total null:",     df.loc[mask_tac,     "std_qty_total"].isna().sum())

df.loc[mask_anadolu | mask_tac, ["product_name_raw","std_qty_total","std_unit","dq_missing_unit"]].head(60)

Anadolu kq std_qty_total null: 0
Tac kq std_qty_total null: 0


C:\Users\Duyghu Mammadova\AppData\Local\Temp\ipykernel_6704\4242864645.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_kq = s.str.contains(r"(\bkq\b|\bkg\b|\bkilo\b|kq\s*çəki)", case=False, na=False)


,product_name_raw,std_qty_total,std_unit,dq_missing_unit
65,Tac Şəkər Tozu 5 kq,5000.0,g,0
73,Tac Kelle Qend 5 kq,5000.0,g,0
154,Tac Buğda Unu 5 kq,5000.0,g,0
162,Anadolu Un kq Çəki,1000.0,g,0
163,Tac Buğda Unu kq,1000.0,g,0
167,Tac Makaron Vermişel kq,1000.0,g,0
168,Tac Makaron Qələm kq,1000.0,g,0
169,Tac Makaron Burma kq,1000.0,g,0
170,Tac Makaron Spiral kq,1000.0,g,0
171,Tac Makaron İncə Boru kq,1000.0,g,0


In [7]:
df.columns

Index(['scrape_run_id', 'scrape_ts_utc', 'snapshot_date', 'market',
       'category_raw', 'product_name_raw', 'barcode', 'price_current_azn',
       'price_old_azn', 'price_current_raw', 'price_old_raw', 'promo_flag',
       'purchasable_balance', 'availability_status', 'unit_info_raw',
       'product_hash', 'dq_missing_barcode', 'dq_invalid_price',
       'dq_missing_purchasable_balance', 'scrape_ts', 'scrape_ts_baku',
       'baku_hour', 'session', 'market_day', 'market_day_session',
       'std_qty_total', 'std_unit', 'pack_multiplier', 'dq_missing_unit',
       'unit_price_1kg', 'unit_price_1l', 'unit_price_raw', 'unit_price_base',
       'unit_price_base_label', 'name_lc', 'category_sub', 'category_group',
       'barcode_str', 'cat_lc', 'category_raw_lc', 'catraw_lc', 'brand',
       'barcode_clean', 'valid_barcode', 'dq_invalid_barcode', 'sku_str',
       'global_product_id_sku', 'std_qty', 'name_norm2', 'identity_string',
       'identity_hash', 'global_product_id', 'dq_missi